# Parametrised PINNs for Water Infiltration in Unsaturated Soils

**Paper:** Gowely, M., Yildiz, A. (2026). *Parametrised PINNs for Water Infiltration in Unsaturated Soils.* Preprint, RWTH Aachen University. https://ssrn.com/abstract=6767300

**Carpeta origen:** `PINNs/1. mecanica de fluidos/Parametrised PINNs for Water Infiltration in.pdf`

## Como se usan las PINNs en este paper

El paper resuelve la **ecuacion de Richardson-Richards (RRE)** 1D que gobierna la infiltracion de agua en suelos no saturados (Eq. 1):

$$\frac{\partial\theta}{\partial t}=\frac{\partial}{\partial z}\Big[K(\psi)\Big(\frac{\partial\psi}{\partial z}+1\Big)\Big] - S(z,t)$$

con el modelo de Gardner (Eq. 2) para las relaciones constitutivas suelo-agua:

$$K(\psi)=K_s\exp(\alpha\psi),\qquad \theta(\psi)=\theta_r+(\theta_s-\theta_r)\exp(\alpha\psi)$$

y condiciones (Eq. 3): condicion inicial de equilibrio de Gardner, contorno inferior Dirichlet (nivel freatico, $\psi=\psi_{lb}$) y contorno superior Neumann (flujo de lluvia/evaporacion prescrito $q(t)$).

**La contribucion central del paper** es transformar la PINN de un *aproximador de un unico escenario* a un **aproximador universal parametrizado**: en vez de fijar los parametros hidraulicos del suelo $(\alpha,K_s,\theta_r,\theta_s,q_B)$ y reentrenar para cada nuevo suelo, estos parametros se agregan como **entradas adicionales de la red** (Resumen Grafico del paper) junto a $(z,t)$, y se muestrean aleatoriamente dentro de un rango durante el entrenamiento. El resultado es una unica red $\psi_\theta(z,t,\alpha,K_s,\theta_r,\theta_s,q_B)$ que, una vez entrenada, evalua **instantaneamente cualquier combinacion de parametros dentro del rango entrenado**, sin reentrenar ni volver a mallar &mdash; validado en el paper contra la solucion analitica de Yuan y Lu (2005) con NSE > 0.99 en 19 de 21 escenarios.

Este cuaderno reproduce fielmente: la RRE con modelo de Gardner, las condiciones inicial/contorno, y el **mecanismo de parametrizacion** (parametros del suelo como entradas adicionales, muestreados aleatoriamente durante el entrenamiento), demostrando que una unica red entrenada generaliza a multiples combinaciones de parametros del suelo.

## Repositorio publico de referencia

El PDF (preprint SSRN) no incluye un repositorio de codigo propio. Como referencia publica del mismo tipo de problema (PINNs para la ecuacion de Richards en suelos no saturados), se usa:

- **Hamza-Kamil/PINN-MPINN-Water-Solute** &mdash; https://github.com/Hamza-Kamil/PINN-MPINN-Water-Solute — PINNs y multi-physics-PINNs para flujo de agua y transporte de solutos en suelos no saturados (TensorFlow).

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Dominio y rangos de parametros hidraulicos (muestreados durante el entrenamiento)

Dominio $z\in[-L,0]$ (nivel freatico en $z=-L$, superficie en $z=0$), $t\in[0,T]$. La condicion inicial de equilibrio hidrostatico sin flujo ($q=0$) satisface exactamente la RRE en estado estacionario para **cualquier** valor de los parametros de Gardner: $\psi_0(z)=\psi_{lb}-(z-z_{lb})$.

In [ ]:
L = 1.0     # profundidad del dominio hasta el nivel freatico (normalizada)
T = 1.0     # horizonte temporal (normalizado)
psi_lb = 0.0  # presion en el nivel freatico (contorno inferior, Eq. 3b)

# Rangos de los 5 parametros hidraulicos parametrizados (Resumen Grafico: p = (alpha, Ks, thetar, thetas, qB))
ranges = {
    'alpha':  (0.5, 2.0),
    'Ks':     (0.05, 0.2),
    'theta_r': (0.05, 0.10),
    'theta_s': (0.35, 0.45),
    'qB':     (-0.05, 0.0),   # flujo superficial: qB<0 representa infiltracion (lluvia)
}

def sample_params(n):
    return {k: torch.rand(n, 1, device=device) * (hi - lo) + lo for k, (lo, hi) in ranges.items()}

def psi0(z):
    """Perfil de equilibrio hidrostatico (q=0): satisface la RRE exactamente para cualquier suelo."""
    return psi_lb - (z - (-L))

## 2. Red PINN parametrizada: entradas $(z,t,\alpha,K_s,\theta_r,\theta_s,q_B)$ (Resumen Grafico del paper)

In [ ]:
class ParametrisedPINN(nn.Module):
    def __init__(self, n_hidden=5, n_neurons=64):
        super().__init__()
        n_in = 7  # z, t, alpha, Ks, theta_r, theta_s, qB
        layers = [nn.Linear(n_in, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, z, t, p):
        # Normalizacion lineal de cada entrada a [0,1] (Eq. de normalizacion del paper, x_norm)
        zn = (z - (-L)) / L
        tn = t / T
        inputs = [zn, tn]
        for k in ['alpha', 'Ks', 'theta_r', 'theta_s', 'qB']:
            lo, hi = ranges[k]
            inputs.append((p[k] - lo) / (hi - lo))
        x = torch.cat(inputs, dim=1)
        return self.net(x)


model = ParametrisedPINN().to(device)


def d_d(f, v):
    return torch.autograd.grad(f, v, grad_outputs=torch.ones_like(f),
                                create_graph=True, retain_graph=True)[0]

## 3. Residuo fisico: RRE + Gardner (Eq. 1-2), evaluado con parametros muestreados aleatoriamente

In [ ]:
N_col, N_bc, N_ic = 2000, 200, 200


def compute_loss():
    # --- Residuo PDE en el interior ---
    p = sample_params(N_col)
    z = (torch.rand(N_col, 1, device=device) * L - L).requires_grad_(True)
    t = (torch.rand(N_col, 1, device=device) * T).requires_grad_(True)
    psi = model(z, t, p)
    theta = p['theta_r'] + (p['theta_s'] - p['theta_r']) * torch.exp(p['alpha'] * psi)
    K = p['Ks'] * torch.exp(p['alpha'] * psi)
    psi_z = d_d(psi, z)
    flux = K * (psi_z + 1)
    flux_z = d_d(flux, z)
    theta_t = d_d(theta, t)
    res_pde = theta_t - flux_z
    loss_pde = torch.mean(res_pde**2)

    # --- Condicion inicial (Eq. 3a): perfil de equilibrio hidrostatico ---
    p_ic = sample_params(N_ic)
    z_ic = torch.rand(N_ic, 1, device=device) * L - L
    t_ic = torch.zeros(N_ic, 1, device=device)
    psi_ic_pred = model(z_ic, t_ic, p_ic)
    loss_ic = torch.mean((psi_ic_pred - psi0(z_ic))**2)

    # --- Contorno inferior (Eq. 3b): Dirichlet, nivel freatico ---
    p_lb = sample_params(N_bc)
    z_lb = torch.full((N_bc, 1), -L, device=device)
    t_lb = torch.rand(N_bc, 1, device=device) * T
    psi_lb_pred = model(z_lb, t_lb, p_lb)
    loss_bc_lower = torch.mean((psi_lb_pred - psi_lb)**2)

    # --- Contorno superior (Eq. 3c): Neumann, flujo prescrito qB ---
    p_ub = sample_params(N_bc)
    z_ub = torch.zeros(N_bc, 1, device=device, requires_grad=True)
    t_ub = torch.rand(N_bc, 1, device=device)
    psi_ub_pred = model(z_ub, t_ub, p_ub)
    K_ub = p_ub['Ks'] * torch.exp(p_ub['alpha'] * psi_ub_pred)
    psi_ub_z = d_d(psi_ub_pred, z_ub)
    flux_ub = K_ub * (psi_ub_z + 1)
    loss_bc_upper = torch.mean((flux_ub - (-p_ub['qB']))**2)

    loss_bc_ic = loss_ic + loss_bc_lower + loss_bc_upper
    total = loss_pde + 10.0 * loss_bc_ic
    return total, loss_pde.item(), loss_bc_ic.item()

## 4. Entrenamiento (una unica red para todo el rango de parametros del suelo)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
for epoch in range(4000):
    optimizer.zero_grad()
    loss, l_pde, l_bc = compute_loss()
    loss.backward()
    optimizer.step()
    history.append(loss.item())
    if epoch % 500 == 0:
        print(f'epoch {epoch:5d} | loss={loss.item():.4e} | pde={l_pde:.4e} | bc+ic={l_bc:.4e}')

## 5. Evaluacion: la MISMA red, sin reentrenar, para varios suelos distintos (el punto central del paper)

In [ ]:
z_plot = torch.linspace(-L, 0, 150, device=device).view(-1, 1)
t_fixed = torch.full_like(z_plot, T)

scenarios = [
    {'alpha': 0.7, 'Ks': 0.08, 'theta_r': 0.06, 'theta_s': 0.38, 'qB': -0.01},
    {'alpha': 1.3, 'Ks': 0.15, 'theta_r': 0.08, 'theta_s': 0.42, 'qB': -0.03},
    {'alpha': 1.8, 'Ks': 0.19, 'theta_r': 0.09, 'theta_s': 0.44, 'qB': -0.045},
]

plt.figure(figsize=(6, 6))
for sc in scenarios:
    p_eval = {k: torch.full_like(z_plot, v) for k, v in sc.items()}
    with torch.no_grad():
        psi_pred = model(z_plot, t_fixed, p_eval)
    label = f"alpha={sc['alpha']}, Ks={sc['Ks']}, qB={sc['qB']}"
    plt.plot(psi_pred.cpu().numpy(), z_plot.cpu().numpy(), label=label)

plt.xlabel('$\\psi$ (presion)')
plt.ylabel('z (profundidad, 0=superficie)')
plt.title(f'Perfil de presion en t={T} para 3 suelos distintos\n(una unica red, sin reentrenar)')
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(6, 4))
plt.semilogy(history)
plt.xlabel('Epoca'); plt.ylabel('Loss (escala log)')
plt.title('Convergencia de la P-PINN')
plt.grid(alpha=0.3)
plt.show()

Las tres curvas se obtienen evaluando **la misma red entrenada una sola vez**, cambiando unicamente los valores de $(\alpha,K_s,\theta_r,\theta_s,q_B)$ que se pasan como entrada -- sin reentrenar ni volver a discretizar el dominio. Esto es exactamente la ventaja practica que reporta el paper: el costo de entrenamiento inicial es mayor (aqui, 5 parametros adicionales en la entrada), pero se amortiza al evaluar instantaneamente cualquier escenario dentro del rango entrenado, en contraste con una PINN convencional que necesitaria reentrenarse por completo para cada combinacion de propiedades del suelo.